# Regression Training

In [ ]:
save_models_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/dif_to_line/"
prefix_name = "regressor_dif_"

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from nba_ou.data_processing.missing_data.handle_missing_data import apply_missing_policy, compute_and_save_train_medians

## Load Data

In [ ]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "training_data_from_2004-10-01_to_2026-01-10.csv"
path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [ ]:
df_stats= df_stats[df_stats['TOTAL_POINTS'] > 130]

In [ ]:
df_stats.shape[0]

In [ ]:
df_stats

In [ ]:
#count nas in TOTA_OVER_UNDER_LINE
nans = df_stats['TOTAL_OVER_UNDER_LINE'].isna().sum()
print(f"Number of NaNs in TOTAL_OVER_UNDER_LINE: {nans}")

In [ ]:
df_stats.dropna(subset=['TOTAL_OVER_UNDER_LINE'], inplace=True)
df_stats = df_stats[df_stats['TOTAL_OVER_UNDER_LINE'] > 100]

In [ ]:
df_stats['SEASON_YEAR'].unique()

In [ ]:
#drop season 2007
df_stats = df_stats[df_stats['SEASON_YEAR'] != 2007]

In [ ]:
df_to_train = df_stats.copy()

In [ ]:
text_columns = df_to_train.select_dtypes(include=['object', 'string']).columns

# Print the names of the columns containing text
# print("Text columns:", list(text_columns))
cols_to_drop = ['SEASON_ID', 'GAME_ID', 'SEASON_TYPE',  'TEAM_ID_TEAM_HOME',
       'TEAM_ID_TEAM_AWAY','IS_OVERTIME']

cols_to_drop += [
    "MATCHUP_TEAM_HOME",
    "GAME_NUMBER_TEAM_HOME",
    "TEAM_CITY_TEAM_AWAY",
    "TEAM_ABBREVIATION_TEAM_AWAY",
    "TEAM_ABBREVIATION_TEAM_HOME",
    "TEAM_CITY_TEAM_HOME",
    "TEAM_NAME_TEAM_AWAY",
    "MATCHUP_TEAM_AWAY",
    'TEAM_NAME_TEAM_HOME'
]
cols_to_drop += list(text_columns)

cols_to_drop.pop(cols_to_drop.index('GAME_DATE'))

df_to_train.drop(columns=cols_to_drop, inplace=True, errors='ignore')


In [ ]:
#Calculate number fo rows with at least 1 NA in total and in percentage
rows_with_na = df_to_train.isna().any(axis=1).sum()
total_rows = df_to_train.shape[0]
percentage_with_na = (rows_with_na / total_rows) * 100
print(f"Rows with at least one NA: {rows_with_na} out of {total_rows} ({percentage_with_na:.2f}%)")

In [ ]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_stats[df_stats[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_stats.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

In [ ]:
# df_to_train.fillna(0, inplace=True)

In [ ]:
print("Applying missing data policy...")
print(f"Data shape before handling missing data: {df_to_train.shape}")

In [ ]:
    # Training time
train_medians = compute_and_save_train_medians(df_to_train)

# Prediction time
df_to_train, report = apply_missing_policy(
    df_to_train,
    train_medians=train_medians,
    current_total_line_col="TOTAL_OVER_UNDER_LINE",
    drop_mode="strict",)

In [ ]:
# report

In [ ]:
print(f"Data shape after handling missing data: {df_to_train.shape}")

In [ ]:
null_rows = df_to_train[df_to_train.isnull().any(axis=1)]
print(f"Total number of null values in the DataFrame: {null_rows.shape[0]}")
print(f"Percentage of null values in the DataFrame: {null_rows.shape[0] / df_to_train.shape[0] * 100:.2f}%")
print(f"Total number of rows in the DataFrame: {df_to_train.shape[0]}")

null_columns = df_to_train.columns[df_to_train.isnull().any()].tolist()
print(f"Columns with null values: {null_columns}")


In [ ]:
df_to_train['LINE_ERROR'] = df_to_train['TOTAL_POINTS'] - df_to_train['TOTAL_OVER_UNDER_LINE']

In [ ]:
df_to_train.drop(columns='GAME_DATE', inplace=True)

## Train / Test

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df_to_train.drop(cols_to_drop + ['LINE_ERROR', 'TOTAL_POINTS'], axis=1, errors='ignore')
y = df_to_train['LINE_ERROR']

In [ ]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=16)

In [ ]:
df_to_train['IS_TRAINING_DATA'] = False

# Mark True for the rows in the training set
df_to_train.loc[X_train.index, 'IS_TRAINING_DATA'] = True
output_name = f"{data_path}/training_data_with_missing_data_handled_from_2004-10-01_to_2026-01-10_dist.csv"
df_to_train.to_csv(output_name, index=False)

In [ ]:
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
# Check number of coulmns
print(f"Number of columns in training set: {X_train.shape[1]}")
print(f"Number of columns in test set: {X_test.shape[1]}")

## Cross-validation

In [ ]:
from sklearn.model_selection import KFold, cross_validate, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error

In [ ]:
# Declare KFold
kf = KFold(n_splits=5, shuffle=True, random_state=16)

In [ ]:
# Custom scorer for over/under betting accuracy when predicting distance to line
def over_under_betting_accuracy(y_true, y_pred):
    """
    Calculate the accuracy of over/under betting decisions when predicting LINE_ERROR.
    
    Since we're predicting LINE_ERROR = TOTAL_POINTS - BETTING_LINE:
    - If LINE_ERROR > 0: game went OVER
    - If LINE_ERROR < 0: game went UNDER
    
    A bet is successful if:
    - We predict OVER (pred > 0) and actual is OVER (true > 0), OR
    - We predict UNDER (pred < 0) and actual is UNDER (true < 0)
    
    Parameters:
    -----------
    y_true : array-like
        Actual LINE_ERROR (actual total points - betting line)
    y_pred : array-like
        Predicted LINE_ERROR (predicted total points - betting line)
    
    Returns:
    --------
    float : Accuracy of betting decisions (0 to 1)
    """
    # Since we're predicting distance to line, we just need to check if signs match
    # np.sign returns -1, 0, or 1
    correct_predictions = np.sign(y_pred) == np.sign(y_true)
    
    # Calculate accuracy
    accuracy = np.mean(correct_predictions)
    
    return accuracy


# Custom scorer class to work with sklearn
class OverUnderScorer:
    """
    Custom scorer that calculates betting accuracy for over/under predictions.
    Works with LINE_ERROR predictions (distance to betting line).
    """
    def __call__(self, estimator, X, y_true):
        """
        Calculate the over/under betting accuracy.
        
        Parameters:
        -----------
        estimator : fitted estimator
            The model to evaluate
        X : DataFrame
            Features
        y_true : array-like
            Actual LINE_ERROR
        
        Returns:
        --------
        float : Betting accuracy score
        """
        y_pred = estimator.predict(X)
        return over_under_betting_accuracy(y_true, y_pred)


# Create a scorer object
over_under_scorer = OverUnderScorer()

In [ ]:
# Declare scores to be used
scoring = {
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(root_mean_squared_error),
    'MAE': make_scorer(mean_absolute_error),
    'OU_Betting_Accuracy': over_under_scorer,
}

In [ ]:
def print_metrics(cv_results):
    for sc in scoring.keys():
        if sc == 'OU_Betting_Accuracy':
            print(f'Train {sc}:', f"{cv_results[f'train_{sc}'].mean():.2%}")
            print(f'Validation {sc}:', f"{cv_results[f'test_{sc}'].mean():.2%}")
        else:
            print(f'Train {sc}:', cv_results[f'train_{sc}'].mean().round(5))
            print(f'Validation {sc}:', cv_results[f'test_{sc}'].mean().round(5))
        print()

In [ ]:
def real_vs_pred(model, X_train, y_train):
    preds = cross_val_predict(model, X_train, y_train, cv=kf, n_jobs=-1)
    x_line = np.arange(y_train.min(), y_train.max())
    plt.scatter(y_train, preds)
    plt.plot(x_line, x_line, color='orange')
    plt.xlabel('Real target')
    plt.ylabel('Predicted target')
    plt.show()

## Baseline

In [ ]:
from sklearn.dummy import DummyRegressor

In [ ]:
season_bl = DummyRegressor(strategy='mean')
cv_results = cross_validate(season_bl, X_train, y_train, cv=kf,
                            scoring=scoring, return_train_score=True)
season_bl.fit(X_train, y_train)
print_metrics(cv_results)

In [ ]:
real_vs_pred(season_bl, X_train, y_train)

In [ ]:
y_pred_baseline_2 = X_train['PTS_SEASON_BEFORE_AVG_TEAM_HOME'] + X_train['PTS_SEASON_BEFORE_AVG_TEAM_AWAY']

# Evaluate manually
mse = mean_squared_error(y_train, y_pred_baseline_2)
mae = mean_absolute_error(y_train, y_pred_baseline_2)
rmse = root_mean_squared_error(y_train, y_pred_baseline_2)
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

In [ ]:
x_line = np.arange(y_train.min(), y_train.max())
plt.scatter(y_train, y_pred_baseline_2)
plt.plot(x_line, x_line, color='orange')
plt.xlabel('Real target')
plt.ylabel('Predicted target')
plt.show()

In [ ]:
# Baseline 3: Predict the betting line (TOTAL_OVER_UNDER_LINE)
y_pred_baseline_3 = X_train['TOTAL_OVER_UNDER_LINE']

# Evaluate
mse = mean_squared_error(y_train, y_pred_baseline_3)
mae = mean_absolute_error(y_train, y_pred_baseline_3)
rmse = root_mean_squared_error(y_train, y_pred_baseline_3)
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")

In [ ]:
x_line = np.arange(y_train.min(), y_train.max())
plt.scatter(y_train, y_pred_baseline_3)
plt.plot(x_line, x_line, color='orange')
plt.xlabel('Real target')
plt.ylabel('Predicted target')
plt.title('Baseline 3: Betting Line Prediction')
plt.show()

## Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
lr = LinearRegression()
cv_results = cross_validate(lr, X_train, y_train, cv=kf,
                            scoring=scoring, return_train_score=True)

lr.fit(X_train, y_train)
print_metrics(cv_results)

In [ ]:
real_vs_pred(lr, X_train, y_train)

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
X_train.shape

In [ ]:
rf = RandomForestRegressor(100, max_depth=5, max_features=0.6, n_jobs=-1, random_state=16)
cv_results = cross_validate(rf, X_train, y_train, cv=kf,
                            scoring=scoring, return_train_score=True, n_jobs=-1)

rf.fit(X_train, y_train)
print_metrics(cv_results)

In [ ]:
import numpy as np
from functools import partial
from xgboost import XGBRegressor
from sklearn.model_selection import cross_validate


def sign_flip_weighted_mse_obj_sklearn_ramped(
    y_true,
    y_pred,
    w_same=0.9,
    w_flip_base=1.0,
    w_flip_extra=0.4,
    edge_scale=6.0,
    eps_push=1.0,
):
    """
    Custom objective for XGBRegressor: signature (y_true, y_pred) -> (grad, hess)

    Loss per sample: w * (y_pred - y_true)^2

    Weights:
      - If |y_true| < eps_push: neutral, w = 1.0 (or set to 0.0 if you want to ignore)
      - Same side: w = w_same (typically 0.85–1.0)
      - Sign flip: w = w_flip_base + w_flip_extra * min(1, |y_true| / edge_scale)
        (penalizes flipping more when the true edge is large, capped)
    """
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    err = y_pred - y_true

    neutral = np.abs(y_true) < eps_push
    wrong = (~neutral) & ((y_true * y_pred) < 0)
    same = (~neutral) & (~wrong)

    ramp = np.minimum(1.0, np.abs(y_true) / edge_scale).astype(np.float32)

    w = np.ones_like(y_true, dtype=np.float32)
    w[same] = np.float32(w_same)
    w[wrong] = np.float32(w_flip_base) + np.float32(w_flip_extra) * ramp[wrong]
    w[neutral] = 1.0  # set to 0.0 if you truly want to ignore near-push examples

    grad = 2.0 * w * err
    hess = 2.0 * w
    return grad, hess

custom_obj = partial(
    sign_flip_weighted_mse_obj_sklearn_ramped,
    w_same=0.9,
    w_flip_base=1.0,
    w_flip_extra=0.4,
    edge_scale=6.0,
    eps_push=1.0,
)


xgb_reg = XGBRegressor(
    max_depth=4,              # reduced depth (helps generalization)
    learning_rate=0.025,
    n_estimators=2000,        # use early stopping ideally; if not, higher n_estimators + reg
    subsample=0.8,            # < 1 helps
    colsample_bytree=0.8,
    min_child_weight=10,      # stronger regularization
    reg_lambda=10.0,          # stronger L2
    reg_alpha=0.1,            # optionally 0.1–1.0
    n_jobs=-2,
    tree_method="hist",
    objective=custom_obj,
    eval_metric="rmse",
    random_state=16, 
)
cv_results = cross_validate(
    xgb_reg,
    X_train, y_train,
    cv=kf,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-2,
    error_score="raise",  # helps debug immediately if anything else fails
)

xgb_reg.fit(X_train, y_train)
print_metrics(cv_results)


## Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from xgboost import XGBRegressor
from sklearn.model_selection import cross_validate

### Randomized Search

### Grid Search

In [ ]:
def print_rs_reg_scores(rs_model, best_idx):
    print(rs_model.best_params_)
    print()
    print('Train MAE:', rs_model.cv_results_['mean_train_MAE'][best_idx].round(5))
    print('Validation MAE:', rs_model.cv_results_['mean_test_MAE'][best_idx].round(5))
    print()
    print('Train MSE:', rs_model.cv_results_['mean_train_MSE'][best_idx].round(5))
    print('Validation MSE:', rs_model.cv_results_['mean_test_MSE'][best_idx].round(5))
    print()
    print('Train RMSE:', rs_model.cv_results_['mean_train_RMSE'][best_idx].round(5))
    print('Validation RMSE:', rs_model.cv_results_['mean_test_RMSE'][best_idx].round(5))
    print()
    print('Train OU Betting Accuracy:', f"{rs_model.cv_results_['mean_train_OU_Betting_Accuracy'][best_idx]:.2%}")
    print('Validation OU Betting Accuracy:', f"{rs_model.cv_results_['mean_test_OU_Betting_Accuracy'][best_idx]:.2%}")

In [ ]:
import os
n_jobs = max(1, os.cpu_count() - 2)
print(f"Number of jobs: {n_jobs}")

In [ ]:
X_train.shape


In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

# Define the base model
xgb_reg = XGBRegressor(
    tree_method="hist",
    objective=custom_obj,
    eval_metric="rmse",   # recommended to set explicitly when using a custom objective
    random_state=16,
)

# Define hyperparameter grid
params = [{
    'n_estimators': [200, 500],  # Similar to GBRT
    'learning_rate': [0.02, 0.04, 0.01],  # Fine-tuning LR
    # 'max_depth': [4, 5],  # Keeping it limited for efficiency
    # 'min_child_weight': [3, 4],  # Similar to min_samples_split in GBRT
    # 'subsample': [0.8, 0.5],  # Control for overfitting
    # 'colsample_bytree': [0.8, 1.0],  # Similar to max_features in GBRT
    'gamma': [0],  # Regularization parameter
}]

# GridSearchCV setup
xgb_gs = GridSearchCV(
    xgb_reg,
    param_grid=params,
    scoring=scoring,
    cv=kf,
    verbose=3,
    refit=lambda x: x["mean_test_OU_Betting_Accuracy"].argmax(),  # Select best model based on MAE
    return_train_score=True,
    n_jobs=-1,
)

# Fit model
xgb_gs.fit(X_train, y_train)

In [ ]:
# Print metrics
xgb_gs.best_params_
print_rs_reg_scores(xgb_gs, xgb_gs.best_index_)

In [ ]:
{'gamma': 0, 'learning_rate': 0.02, 'n_estimators': 500}


# Pair LR and n_estimators to avoid wasting compute on mismatched combos
param_grid = [
    # Regime A: what your current data seems to like (0.02, ~200) plus nearby
    {
        "learning_rate": [0.02],
        "n_estimators": [500],
        "max_depth": [3, 4, 5,6],
        "min_child_weight": [3, 4,5,6],
        "subsample": [0.5, 0.7],
        "colsample_bytree": [0.6, 0.8, 1.0],
        "gamma": [0, 0.1],
        "reg_lambda": [1, 2],
        "reg_alpha": [0, 0.1],
    }
   
]

xgb_gs = GridSearchCV(
    estimator=xgb_reg,
    param_grid=param_grid,
    scoring=scoring,
    cv=kf,
    verbose=3,
    refit=lambda x: x["mean_test_OU_Betting_Accuracy"].argmax(),  # as you are doing
    return_train_score=True,
    n_jobs=-1,
)

xgb_gs.fit(X_train, y_train)


In [ ]:
# Print metrics
xgb_gs.best_params_
print_rs_reg_scores(xgb_gs, xgb_gs.best_index_)

In [ ]:
{'colsample_bytree': 0.6, 'gamma': 0, 'learning_rate': 0.02, 'max_depth': 6, 'min_child_weight': 3, 'n_estimators': 500, 'reg_alpha': 0, 'reg_lambda': 2, 'subsample': 0.7}
xgb_reg = XGBRegressor(
    tree_method="hist",
    objective=custom_obj,
    eval_metric="rmse",   # recommended to set explicitly when using a custom objective
    random_state=16,
)


# Pair LR and n_estimators to avoid wasting compute on mismatched combos
param_grid = [
    # Regime A: what your current data seems to like (0.02, ~200) plus nearby
    {
        "learning_rate": [0.02, 0.025, 0.017,0.1],
        "n_estimators": [500,600,700],
        "max_depth": [6],
        "min_child_weight": [3],
        "subsample": [0.7],
        "colsample_bytree": [0.6],
        "gamma": [0],
        "reg_lambda": [2],
        "reg_alpha": [0],
    }
]

xgb_gs = GridSearchCV(
    estimator=xgb_reg,
    param_grid=param_grid,
    scoring=scoring,
    cv=kf,
    verbose=3,
    refit=lambda x: x["mean_test_OU_Betting_Accuracy"].argmax(),  # as you are doing
    return_train_score=True,
    n_jobs=-2,
)

xgb_gs.fit(X_train, y_train)


In [ ]:
# Print metrics
xgb_gs.best_params_
print_rs_reg_scores(xgb_gs, xgb_gs.best_index_)

In [ ]:
{'colsample_bytree': 0.6, 'gamma': 0, 'learning_rate': 0.02, 'max_depth': 6, 'min_child_weight': 3, 'n_estimators': 600, 'reg_alpha': 0, 'reg_lambda': 2, 'subsample': 0.7}


param_grid = [
    {
        "learning_rate": [0.02],
        "n_estimators": [650,600,550],
        "max_depth": [6],
        "min_child_weight": [3],
        "subsample": [0.7, 0.6],
        "colsample_bytree": [0.6],
        "gamma": [0],
        "reg_lambda": [2],
        "reg_alpha": [0],
    }
]

xgb_gs = GridSearchCV(
    estimator=xgb_reg,
    param_grid=param_grid,
    scoring=scoring,
    cv=kf,
    verbose=1,
    refit=lambda x: x["mean_test_OU_Betting_Accuracy"].argmax(),  # as you are doing
    return_train_score=True,
    n_jobs=-2,
)

xgb_gs.fit(X_train, y_train)


In [ ]:
# Print metrics
xgb_gs.best_params_
print_rs_reg_scores(xgb_gs, xgb_gs.best_index_)

In [ ]:
import pickle

# Get the best model
best_model = xgb_gs.best_estimator_
#get_date
import datetime
now = datetime.datetime.now()
date_today = now.strftime("%Y-%m-%d")

name = f"best_xgboost_{prefix_name}{date_today}.pkl"
name_config = f"best_xgboost_{prefix_name}config_{date_today}.pkl"
# Save the best model
with open(save_models_path+name, "wb") as f:
    pickle.dump(best_model, f)

# Save the best model configuration
with open(save_models_path+name_config, "wb") as f:
    pickle.dump(xgb_gs.best_params_, f)

print("Best model saved as", name)

In [ ]:
# name = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/best_xgboost_regressor_total_2026-01-20.pkl"
import pickle
#load model
with open(save_models_path+name, "rb") as f:
    best_model = pickle.load(f)

In [ ]:
feature_importances = best_model.feature_importances_
important_features = np.argsort(feature_importances)[::-1]  
feature_importances = pd.DataFrame({
    'Feature': X_train.columns[important_features],
    'Importance': feature_importances[important_features]
}).sort_values(by="Importance", ascending=False)
feature_importances

In [ ]:
n_columns = X_train.shape[1]
only_important_features =  X_train.columns[important_features][:int(n_columns * 0.667)]

In [ ]:
#repeat the analysis with only important features
xgb_reg_only_import = XGBRegressor(**best_model.get_params())

cv_results_important = cross_validate(
    xgb_reg_only_import, 
    X_train[only_important_features], y_train, 
    cv=kf, 
    scoring=scoring,      # Use your custom scoring or e.g. 'neg_mean_absolute_error'
    return_train_score=True,
    n_jobs=-1
)
# Train final model on full train set
xgb_reg_only_import.fit(X_train[only_important_features], y_train)

# Print metrics
print_metrics(cv_results_important)

In [ ]:
best_model.get_params()


In [ ]:

{'colsample_bytree': 0.6, 'gamma': 0, 'learning_rate': 0.025, 'max_depth': 4, 'min_child_weight': 4, 'n_estimators': 600, 'reg_alpha': 0.1, 'reg_lambda': 1, 'subsample': 0.6}
# Define the base model
xgb_reg = XGBRegressor(random_state=16, n_jobs=10)

# Define hyperparameter grid
params = [{
    'n_estimators': [600],  # Similar to GBRT
    'learning_rate': [0.02, 0.025],  # Fine-tuning LR
    'max_depth': [3,4,5,6],  # Keeping it limited for efficiency
    'min_child_weight': [3,4,5],  # Similar to min_samples_split in GBRT
    'subsample': [0.6, 0.7, 0.5],  # Control for overfitting
    'colsample_bytree': [0.8, 0.6],  # Similar to max_features in GBRT
    'gamma': [0],
    'reg_alpha': [0.1],
    'reg_lambda': [1],  # Regularization parameter
}]

# GridSearchCV setup
xgb_gs_import = GridSearchCV(
    xgb_reg,
    param_grid=params,
    scoring=scoring,
    cv=kf,
    verbose=3,
    refit=lambda x: x["mean_test_OU_Betting_Accuracy"].argmax(),  # as you are doing
    return_train_score=True,
    n_jobs= 10
)

# Fit model
xgb_gs_import.fit(X_train[only_important_features], y_train)

In [ ]:
# Print metrics
xgb_gs_import.best_params_
print_rs_reg_scores(xgb_gs_import, xgb_gs_import.best_index_)

In [ ]:

{'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.02, 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 600, 'reg_alpha': 0.1, 'reg_lambda': 1, 'subsample': 0.6}

# Define the base model
xgb_reg = XGBRegressor(random_state=16, n_jobs=-2)

# Define hyperparameter grid
params = [{
    'n_estimators': [600, 750, 1000],  # Similar to GBRT
    'learning_rate': [0.02, 0.017, 0.023],  
    'max_depth': [5],  # Keeping it limited for efficiency
    'min_child_weight': [ 3],  # Similar to min_samples_split in GBRT
    'subsample': [0.6],  # Control for overfitting
    'colsample_bytree': [0.8],  # Similar to max_features in GBRT
    'gamma': [0],
    'reg_alpha': [0.1],
    'reg_lambda': [1], 
    # 'colsample_bylevel': [0.7, 0.8,  1.0],
    # 'colsample_bynode': [0.8, 0.9, 1.0],# Regularization parameter
}]

# GridSearchCV setup
xgb_gs_import = GridSearchCV(
    xgb_reg,
    param_grid=params,
    scoring=scoring,
    cv=kf,
    verbose=3,
    refit=lambda x: x["mean_test_OU_Betting_Accuracy"].argmax(),  # as you are doing
    return_train_score=True,
    n_jobs=-2
)

# Fit model
xgb_gs_import.fit(X_train[only_important_features], y_train)


In [ ]:
xgb_gs_import.best_params_
print_rs_reg_scores(xgb_gs_import, xgb_gs_import.best_index_)

In [ ]:

{'colsample_bytree': 0.8, 'gamma': 0, 'learning_rate': 0.02, 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 600, 'reg_alpha': 0.1, 'reg_lambda': 1, 'subsample': 0.6}

# Define the base model
xgb_reg = XGBRegressor(random_state=16, n_jobs=-2)

# Define hyperparameter grid
params = [{
    'n_estimators': [600, 650],  # Similar to GBRT
    'learning_rate': [0.02],  
    'max_depth': [5],  # Keeping it limited for efficiency
    'min_child_weight': [ 3],  # Similar to min_samples_split in GBRT
    'subsample': [0.6],  # Control for overfitting
    'colsample_bytree': [0.8],  # Similar to max_features in GBRT
    'gamma': [0],
    'reg_alpha': [0.1],
    'reg_lambda': [1], 
    'colsample_bylevel': [0.7, 0.8,  1.0],
    'colsample_bynode': [0.8, 0.9, 1.0],# Regularization parameter
}]

# GridSearchCV setup
xgb_gs_import = GridSearchCV(
    xgb_reg,
    param_grid=params,
    scoring=scoring,
    cv=kf,
    verbose=3,
    refit=lambda x: x["mean_test_OU_Betting_Accuracy"].argmax(),  # as you are doing
    return_train_score=True,
    n_jobs=-2
)

# Fit model
xgb_gs_import.fit(X_train[only_important_features], y_train)


In [ ]:
xgb_gs_import.best_params_
print_rs_reg_scores(xgb_gs_import, xgb_gs_import.best_index_)

In [ ]:
import pickle

# Get the best model
best_model = xgb_gs_import.best_estimator_
#get_date
import datetime
now = datetime.datetime.now()
date_today = now.strftime("%Y-%m-%d")
name = f"best_xgboost_{prefix_name}important_{date_today}.pkl"
name_config = f"best_xgboost_{prefix_name}important_config_{date_today}.pkl"
# Save the best model
with open(save_models_path+ name, "wb") as f:
    pickle.dump(best_model, f)

print("Best model saved as", name)

# Train whole data

In [ ]:
name_total = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/dif_to_line/best_xgboost_regressor_dif_2026-01-22.pkl"

In [ ]:
import pickle

with open(name_total, "rb") as f:
    best_model_total_features = pickle.load(f)

In [ ]:
from xgboost import XGBRegressor

production_model = XGBRegressor(**best_model_total_features.get_params())
production_model.fit(X, y)

In [ ]:
# insert in the model before the .pickle full_dataset 
production_model_filename = f"{name_total.replace('.pkl', '_production.pkl')}"
with open(production_model_filename, "wb") as f:
    pickle.dump(production_model, f)

print("Production-ready model saved as", production_model_filename)

In [ ]:
name_important = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points_bet_accuracy/best_xgboost_regressor_total_bet_important_2026-01-21.pkl"

In [ ]:
import pickle

with open(name_important, "rb") as f:
    best_model_important_features = pickle.load(f)

In [ ]:
from xgboost import XGBRegressor
trained_features = best_model_important_features.feature_names_in_

production_model_important = XGBRegressor(
    **best_model_important_features.get_params()
)

production_model_important.fit(
    X[trained_features],
    y
)

In [ ]:
# insert in the model before the .pickle full_dataset 
production_model_important_filename = f"{name_important.replace('.pkl', '_production.pkl')}"
with open(production_model_important_filename, "wb") as f:
    pickle.dump(production_model_important, f)

print("Production-ready model (important features) saved as", production_model_important_filename)

In [ ]:
df_until_25 = df_to_train[df_to_train['SEASON_YEAR'] < 2025]


X2 = df_until_25.drop(cols_to_drop + ['TOTAL_POINTS'], axis=1, errors='ignore')
y2 = df_until_25['TOTAL_POINTS']

In [ ]:
from xgboost import XGBRegressor

production_model = XGBRegressor(**best_model_total_features.get_params())
production_model.fit(X, y)

In [ ]:
# insert in the model before the .pickle full_dataset 
production_model_filename = f"{name_total.replace('.pkl', '_production_till_2025.pkl')}"
with open(production_model_filename, "wb") as f:
    pickle.dump(production_model, f)

print("Production-ready model saved as", production_model_filename)